# OPTIMIZED: Top 100 Features Only + Aggressive Simplification
Push toward positive R² by using only the BEST 100 features with minimal noise

**Strategy:**
1. Use ONLY top 100 most important features (from previous analysis)
2. Simpler model: max_depth=4, lower regularization
3. Direct ensemble (no complex tuning)
4. Goal: Cross R²=0 on test

**Why this works:** Fewer features = less noise = closer to zero = easier to go positive

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import gc
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("OPTIMIZED: TOP 100 FEATURES + SIMPLIFIED MODEL")
print("="*70)

# ============== LOAD DATA ==============
print("\n[1/3] Loading data...")
train_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/train.parquet')
test_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/test.parquet')

def compress_dtypes(df):
    for col in df.columns:
        if df[col].dtype != 'object':
            c_min, c_max = df[col].min(), df[col].max()
            if str(df[col].dtype)[:3] == 'int':
                if c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df

train_df = compress_dtypes(train_df)
test_df = compress_dtypes(test_df)
train_df = train_df.fillna(train_df.median())
test_df = test_df.fillna(test_df.median())

y_train = train_df['TARGET'].values.copy()
test_ids = test_df['ID'].values.copy()

X_train = train_df.drop(['ID', 'TARGET'], axis=1)
X_test = test_df.drop(['ID'], axis=1)

const_cols = list(set(X_train.columns[X_train.nunique() <= 1]) | set(X_test.columns[X_test.nunique() <= 1]))
X_train = X_train.drop(const_cols, axis=1, errors='ignore')
X_test = X_test.drop(const_cols, axis=1, errors='ignore')

common_cols = sorted(list(set(X_train.columns) & set(X_test.columns)))
X_train = X_train[common_cols].copy()
X_test = X_test[common_cols].copy()

del train_df, test_df
gc.collect()
print(f"✓ Train: {X_train.shape}, Test: {X_test.shape}")

# ============== EXTRACT FEATURE IMPORTANCE ==============
print("\n[2/3] Extracting feature importance...")

xgb_params = {
    'objective': 'reg:squarederror',
    'max_depth': 5,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'lambda': 1.0,
    'alpha': 0.5,
    'verbosity': 0
}

model_importance = xgb.XGBRegressor(**xgb_params, n_estimators=300, random_state=42)
model_importance.fit(X_train, y_train, verbose=False)

importance_dict = model_importance.get_booster().get_score(importance_type='weight')
importance_df = pd.DataFrame(list(importance_dict.items()), columns=['feature', 'importance'])
importance_df = importance_df.sort_values('importance', ascending=False).reset_index(drop=True)

print(f"✓ Feature importance extracted")
print(f"\nTop 20 Important Features:")
print(importance_df.head(20).to_string())

# SELECT TOP 100 FEATURES ONLY
top_100_features = importance_df.head(100)['feature'].tolist()
X_train_opt = X_train[top_100_features]
X_test_opt = X_test[top_100_features]

print(f"\n✓ Selected TOP 100 features (removing {len(X_train) - 100} noisy features)")

del model_importance
gc.collect()

# ============== TRAIN OPTIMIZED ENSEMBLE ==============
print("\n[3/3] Training optimized ensemble (5 seeds for diversity)...\n")

seeds = [42, 123, 456, 789, 999]
ensemble_preds = np.zeros(len(X_test_opt))
all_scores = []

# MORE AGGRESSIVE SIMPLIFICATION
xgb_params_opt = {
    'objective': 'reg:squarederror',
    'max_depth': 4,              # REDUCED from 5 (simpler trees)
    'learning_rate': 0.04,       # REDUCED from 0.05 (slower learning)
    'subsample': 0.85,           # UP from 0.8 (more stable samples)
    'colsample_bytree': 0.85,    # UP from 0.8 (more stable features)
    'lambda': 3.0,               # UP from 2.0 (more L2 regularization)
    'alpha': 1.5,                # UP from 1.0 (more L1 regularization)
    'verbosity': 0
}

for seed_idx, seed in enumerate(seeds, 1):
    print(f"  Model {seed_idx}/5 (seed={seed}):")
    
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    fold_indices = list(kf.split(X_train_opt))
    
    test_preds = np.zeros(len(X_test_opt))
    fold_scores = []
    
    for fold_num, (train_idx, val_idx) in enumerate(fold_indices, 1):
        X_tr, X_val = X_train_opt.iloc[train_idx], X_train_opt.iloc[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        model = xgb.XGBRegressor(**xgb_params_opt, n_estimators=350, random_state=seed)
        model.fit(X_tr, y_tr, verbose=False)
        
        test_preds += model.predict(X_test_opt) / 5
        r2 = r2_score(y_val, model.predict(X_val))
        fold_scores.append(r2)
        
        del model, X_tr, X_val, y_tr, y_val
        gc.collect()
    
    avg_r2 = np.mean(fold_scores)
    all_scores.append(avg_r2)
    ensemble_preds += test_preds / 5  # Average across 5 models
    print(f"    Avg R² (5-fold): {avg_r2:.6f}")

print(f"\n✓ Optimized ensemble trained (5 models averaged)")
print(f"✓ Individual model scores: {[f'{s:.6f}' for s in all_scores]}")
print(f"✓ Ensemble Avg R²: {np.mean(all_scores):.6f}")

# ============== CREATE SUBMISSION ==============
print(f"\n[4/4] Creating submission...")
submission = pd.DataFrame({'ID': test_ids, 'TARGET': ensemble_preds})
print(f"✓ Submission shape: {submission.shape}")
print(f"✓ Target - Mean: {submission['TARGET'].mean():.6f}, Std: {submission['TARGET'].std():.6f}")
print(f"✓ Target - Min: {submission['TARGET'].min():.6f}, Max: {submission['TARGET'].max():.6f}")

submission.to_csv('submission_top100_optimized.csv', index=False)
print(f"✓ Saved: submission_top100_optimized.csv")

print("\n" + "="*70)
print("TOP 100 OPTIMIZED - READY FOR KAGGLE")
print("="*70)
print(f"✓ Strategy: Only top 100 most important features")
print(f"✓ Removed: {len(X_train) - 100} noisy features")
print(f"✓ Simplified: max_depth=4, stronger regularization")
print(f"✓ Ensemble: 5 seeds with 5-fold CV each (better diversity)")
print(f"✓ Goal: Push test R² toward positive")
print("="*70)